In [1]:
import pandas as pd
from pathlib import Path
import sqlite3

con = sqlite3.connect('omop.db')
cur = con.cursor()
sus_vocab_path = Path('vocabularies')
list(sus_vocab_path.glob('*.csv'))[0].stem.lower()

for file_path in sus_vocab_path.glob('*.csv'):
    table_name = file_path.stem.lower()
    if table_name == 'source_to_concept_map':
        df = pd.read_csv(file_path, sep=',', dtype=str)
    else:
        df = pd.read_csv(file_path, sep='\t', dtype=str)
    df.to_sql(table_name, con, index=False, if_exists='replace')

In [2]:
cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cur.fetchall()
print(tables)

[('concept',), ('concept_ancestor',), ('concept_class',), ('concept_relationship',), ('concept_synonym',), ('domain',), ('drug_strength',), ('relationship',), ('source_to_concept_map',), ('vocabulary',)]


In [3]:
cur.execute("SELECT COUNT(*) FROM concept;").fetchall()

[(1887964,)]

In [16]:
vocabulary_id = "SUS"


In [50]:
sql_prompt = f"""
SELECT *
FROM source_to_concept_map c
"""

df = pd.read_sql(sql_prompt, con)
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1463 entries, 0 to 1462
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   source_code              1463 non-null   object
 1   source_concept_id        1447 non-null   object
 2   source_vocabulary_id     1447 non-null   object
 3   source_code_description  1447 non-null   object
 4   target_concept_id        1447 non-null   object
 5   target_vocabulary_id     1447 non-null   object
 6   valid_start_date         1447 non-null   object
 7   valid_end_date           1447 non-null   object
 8   invalid_reason           0 non-null      object
 9   domain_id                1447 non-null   object
dtypes: object(10)
memory usage: 114.4+ KB
None


,source_code,source_concept_id,source_vocabulary_id,source_code_description,target_concept_id,target_vocabulary_id,valid_start_date,valid_end_date,invalid_reason,domain_id
0,Crizotinibe,0,SUS,CRIZOTINIBE,40242675,RxNorm,1970-01-01,2099-12-31,None,Drug
1,Cemiplimabe,0,SUS,Cemeiplimab,35200783,RxNorm,1970-01-01,2099-12-31,None,Drug
2,Atezolizumabe,0,SUS,Altezolizumab,42629079,RxNorm,1970-01-01,2099-12-31,None,Drug
3,Panitumumabe,0,SUS,Panitumumab,19100985,RxNorm,1970-01-01,2099-12-31,None,Drug
4,Alfapoetina,0,SUS,Alfapoetin,1301125,RxNorm,1970-01-01,2099-12-31,None,Drug


In [61]:
df.sample(10)

,source_code,source_concept_id,source_vocabulary_id,source_code_description,target_concept_id,target_vocabulary_id,valid_start_date,valid_end_date,invalid_reason,domain_id
794,ELIGAD,0,SUS,Leuprooriline,1351541,RxNorm,1970-01-01,2099-12-31,None,Drug
1420,0506010112,0,SUS,FOLLOW-UP OF PATIENT AFTER LIVER TRANSPLANT,4081759,SNOMED,1970-01-01,2099-12-31,None,Procedure
439,0417010010,0,SUS,Anesthesia for cesarean section,4171820,SNOMED,1970-01-01,2099-12-31,None,Procedure
934,AC ZOLEDRONICO,0,SUS,Ácido zoledrônico monoidratado,1524674,RxNorm,1970-01-01,2099-12-31,None,Drug
103,Amifostina,0,SUS,Amipostin,1350040,RxNorm,1970-01-01,2099-12-31,None,Drug
1434,0403070171,0,SUS,TREATMENT OF ACUTE ISCHEMIC STROKE WITH MECHAN...,43530669,SNOMED,1970-01-01,2099-12-31,None,Procedure
695,0604360029,0,SUS,atorvastatin 20 MG Oral Tablet,19123592,RxNorm,1970-01-01,2099-12-31,None,Drug
853,CARBOINA + TAXO,0,SUS,Carboplatin + Paclitaxel,1344905,RxNorm,1970-01-01,2099-12-31,None,Drug
937,BFM 2009,0,SUS,Metotrexato,1305058,RxNorm,1970-01-01,2099-12-31,None,Drug
935,XELOX,0,SUS,Oxaliplatina,1318011,RxNorm,1970-01-01,2099-12-31,None,Drug


In [17]:
# Conceitos mapeados do SIGTAP para o SNOMED

sql_prompt = f"""
SELECT 
    c.concept_id AS concept_id_1, 
    c.concept_name AS concept_name_1, 
    c.vocabulary_id AS vocabulary_id_1, 
    cr.relationship_id, 
    c2.concept_id  AS concept_id_2, 
    c2.concept_name AS concept_name_2, 
    c2.vocabulary_id AS vocabulary_id_2
FROM concept c
JOIN concept_relationship cr ON c.concept_id = cr.concept_id_1
JOIN concept c2 ON c2.concept_id = cr.concept_id_2
WHERE cr.relationship_id = 'Maps to'
    AND c.vocabulary_id = '{vocabulary_id}'
"""

df = pd.read_sql(sql_prompt, con)
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4523 entries, 0 to 4522
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   concept_id_1     4523 non-null   object
 1   concept_name_1   4523 non-null   object
 2   vocabulary_id_1  4523 non-null   object
 3   relationship_id  4523 non-null   object
 4   concept_id_2     4523 non-null   object
 5   concept_name_2   4523 non-null   object
 6   vocabulary_id_2  4523 non-null   object
dtypes: object(7)
memory usage: 247.5+ KB
None


,concept_id_1,concept_name_1,vocabulary_id_1,relationship_id,concept_id_2,concept_name_2,vocabulary_id_2
0,35703035,BODY PRACTICE / PHYSICAL ACTIVITY IN GROUP,SUS,Maps to,4160730,Physical activity assessment,SNOMED
1,35703036,COLLECTIVE ACTION OF TOPICAL APPLICATION OF FL...,SUS,Maps to,4126461,Topical application of tooth medicament,SNOMED
2,35703037,COLLECTIVE ACTION OF FLUORED BUNCH,SUS,Maps to,4126591,Topical application of fluoride - tooth,SNOMED
3,35703038,COLLECTIVE ACTION OF SUPERVISED DENTAL BRUSH,SUS,Maps to,42872816,Brushing of teeth,SNOMED
4,35703039,APPLICATION OF CARIOSTÁTICO (PER DENT),SUS,Maps to,4040556,Procedure on tooth,SNOMED


In [18]:
df['vocabulary_id_2'].value_counts()

vocabulary_id_2
SNOMED    4173
RxNorm     245
LOINC      105
Name: count, dtype: int64

In [19]:
df['concept_id_1'].value_counts()

concept_id_1
35705952    4
35706314    4
35706955    4
35705447    3
35705448    3
           ..
35707390    1
35707391    1
35707402    1
35707405    1
35707378    1
Name: count, Length: 4117, dtype: int64

In [20]:
(df['concept_id_1'].value_counts()).sum()

np.int64(4523)

In [21]:
df[df['concept_id_1'] == '35706955']

,concept_id_1,concept_name_1,vocabulary_id_1,relationship_id,concept_id_2,concept_name_2,vocabulary_id_2
4107,35706955,OMBITASVIR - 12.5 MG / VERUPREVIR 75 MG / RITO...,SUS,Maps to,1748921,ritonavir,RxNorm
4108,35706955,OMBITASVIR - 12.5 MG / VERUPREVIR 75 MG / RITO...,SUS,Maps to,45892552,ombitasvir,RxNorm
4109,35706955,OMBITASVIR - 12.5 MG / VERUPREVIR 75 MG / RITO...,SUS,Maps to,45892554,paritaprevir,RxNorm
4110,35706955,OMBITASVIR - 12.5 MG / VERUPREVIR 75 MG / RITO...,SUS,Maps to,45892558,dasabuvir,RxNorm


In [22]:
df[df['concept_id_1'] == '35706955'].iloc[0]['concept_name_1']

'OMBITASVIR - 12.5 MG / VERUPREVIR 75 MG / RITONAVIR 50 MG PER TABLET (WITH 02 TABLETS COATED) + DASABUVIR 250 MG PER TABLET (WITH 02 TABLETS COATED).'

In [23]:
# Conceitos do SIGTAP que não possuem mapeamento para standard concept

sql_prompt = f"""
SELECT c.concept_id, c.concept_name, c.concept_code, c.vocabulary_id, cr.relationship_id
FROM concept c
LEFT JOIN concept_relationship cr
  ON c.concept_id = cr.concept_id_1
  AND cr.relationship_id = 'Maps to'
WHERE c.vocabulary_id = '{vocabulary_id}'
  AND cr.concept_id_2 IS NULL;
"""

df = pd.read_sql(sql_prompt, con)
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 476 entries, 0 to 475
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   concept_id       476 non-null    object
 1   concept_name     476 non-null    object
 2   concept_code     476 non-null    object
 3   vocabulary_id    476 non-null    object
 4   relationship_id  0 non-null      object
dtypes: object(5)
memory usage: 18.7+ KB
None


,concept_id,concept_name,concept_code,vocabulary_id,relationship_id
0,35702939,EDUCATIONAL ACTIVITY / GROUP ORIENTATION IN BA...,0101010010,SUS,None
1,35703028,EDUCATIONAL ACTIVITY / GROUP ORIENTATION IN TH...,0101010028,SUS,None
2,35702926,COLLECTIVE ACTION FOR ANIMAL EXAMINATION WITH ...,0101020040,SUS,None
3,35703049,ADMINISTRATION OF VITAMIN A,0101040059,SUS,None
4,35702879,OFFICE OF MASSAGE / SELF-MASSAGE,0101050054,SUS,None


In [24]:
sql_prompt = f"""
SELECT c.concept_id, c.concept_name, c.concept_code, c.vocabulary_id
FROM concept c
WHERE c.vocabulary_id = '{vocabulary_id}'
"""

df = pd.read_sql(sql_prompt, con)
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4593 entries, 0 to 4592
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   concept_id     4593 non-null   object
 1   concept_name   4593 non-null   object
 2   concept_code   4593 non-null   object
 3   vocabulary_id  4593 non-null   object
dtypes: object(4)
memory usage: 143.7+ KB
None


,concept_id,concept_name,concept_code,vocabulary_id
0,35702939,EDUCATIONAL ACTIVITY / GROUP ORIENTATION IN BA...,0101010010,SUS
1,35703028,EDUCATIONAL ACTIVITY / GROUP ORIENTATION IN TH...,0101010028,SUS
2,35703035,BODY PRACTICE / PHYSICAL ACTIVITY IN GROUP,0101010036,SUS
3,35703036,COLLECTIVE ACTION OF TOPICAL APPLICATION OF FL...,0101020015,SUS
4,35703037,COLLECTIVE ACTION OF FLUORED BUNCH,0101020023,SUS


In [25]:
# Conceitos mapeados do SIGTAP para o SNOMED

sql_prompt = f"""
SELECT *
FROM concept c
WHERE c.standard_concept = 'S'
    AND c.vocabulary_id IN ('SNOMED', 'RxNorm', 'LOINC');
"""

df = pd.read_sql(sql_prompt, con)
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 619382 entries, 0 to 619381
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   concept_id        619382 non-null  object
 1   concept_name      619378 non-null  object
 2   domain_id         619382 non-null  object
 3   vocabulary_id     619382 non-null  object
 4   concept_class_id  619382 non-null  object
 5   standard_concept  619382 non-null  object
 6   concept_code      619382 non-null  object
 7   valid_start_date  619382 non-null  object
 8   valid_end_date    619382 non-null  object
 9   invalid_reason    0 non-null       object
dtypes: object(10)
memory usage: 47.3+ MB
None


,concept_id,concept_name,domain_id,vocabulary_id,concept_class_id,standard_concept,concept_code,valid_start_date,valid_end_date,invalid_reason
0,42538812,Somatic hallucination,Condition,SNOMED,Clinical Finding,S,762620006,20180131,20991231,None
1,4084170,Non-allergic anaphylaxis caused by whole blood,Condition,SNOMED,Disorder,S,241944009,20020131,20991231,None
2,4085530,Unformed visual hallucinations,Condition,SNOMED,Clinical Finding,S,247733004,20020131,20991231,None
3,4085038,Formed visual hallucinations,Condition,SNOMED,Clinical Finding,S,247734005,20020131,20991231,None
4,4085531,Scenic visual hallucinations,Condition,SNOMED,Clinical Finding,S,247735006,20020131,20991231,None


In [26]:
df['vocabulary_id'].value_counts()

vocabulary_id
SNOMED    346648
RxNorm    154355
LOINC     118379
Name: count, dtype: int64